In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import os
import cstarpy.integration
COMPOUND_FILE = "cd8_limma_merged_filtered_targets_ic50_dpd.csv"
SELECTION     = "top4_bottom4_selected_modules_drugs_ic50_btla.csv"   # from prep notebook
out_dir       = "02_outputs"
os.makedirs(out_dir, exist_ok=True)

# Load the collapsed-module selection built in prep (module column already exists)
drug_gene = pd.read_csv(SELECTION).rename(columns={"compound_name": "drug"})

modules  = drug_gene["module"].drop_duplicates().tolist()
exp_list = drug_gene["drug"].drop_duplicates().tolist()

print(f"Modules ({len(modules)}): {modules}")
print(f"Experiments ({len(exp_list)}): {exp_list}")

Modules (7): ['BCL', 'BRD', 'HDAC', 'IGF1R', 'JAK', 'MDM2', 'MTOR']
Experiments (14): ['navitoclax', 'I-BET 762', 'Apicidin', 'belinostat', 'panobinostat', 'BMS-536924', 'AT9283', 'TG-101348', 'CYT-387', 'ruxolitinib', 'Serdemetan', 'Sapanisertib', 'Temsirolimus', 'Deforolimus']


In [2]:
df = pd.read_csv(COMPOUND_FILE)

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

x_df = df_avg.pivot_table(
    index="gene", columns="compound_name", values="logFC_thresh", aggfunc="first"
).fillna(0)[exp_list]

genes = x_df.index.tolist()
x     = x_df.values
print(f"x (expression) shape: {x.shape}  (genes × experiments)")

x (expression) shape: (15045, 14)  (genes × experiments)


In [3]:
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

inhib_conc_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
ic50_matrix_top4_bottom4       = np.ones((len(modules), len(exp_list))) * np.inf   # inf → g=1
gamma_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)))            # 0 = inhibitor, >0 = activator
valid_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)), dtype=bool)  # has real dose+IC50 data

GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"

for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        dose = dose_info.get(row["drug"], np.nan)
        ic50_raw = df[df["compound_name"] == row["drug"]]["IC50_nM"]
        ic50_nm = ic50_raw.iloc[0] if len(ic50_raw) else np.nan
        ic50 = ic50_nm / 1000 if pd.notna(ic50_nm) else np.nan
        mech = mech_info.get(row["drug"], "Inhibitor")
        if pd.notna(dose) and pd.notna(ic50):
            inhib_conc_matrix_top4_bottom4[i, j] = dose
            ic50_matrix_top4_bottom4[i, j]       = ic50
            gamma_matrix_top4_bottom4[i, j]      = GAMMA if mech == "Activator" else 0.0
            valid_matrix_top4_bottom4[i, j]      = True
        else:
            print(f"WARNING: missing dose/IC50 for module={row['module']} drug={row['drug']} "
                  f"— excluded from fit (not treated as 'no effect')")

# y_true = (1 + gamma_matrix * dose/IC50) / (1 + dose/IC50)   [gamma=0 collapses to inhibitor form]
dratio_top4_bottom4 = inhib_conc_matrix_top4_bottom4 / ic50_matrix_top4_bottom4
y_true_top4_bottom4 = np.where(
    valid_matrix_top4_bottom4,
    (1 + gamma_matrix_top4_bottom4 * dratio_top4_bottom4) / (1 + dratio_top4_bottom4),
    1.0,
)

print(f"y_true (activity g) shape: {y_true_top4_bottom4.shape}")
print(pd.DataFrame(y_true_top4_bottom4, index=modules, columns=exp_list).round(3).to_string())


y_true (activity g) shape: (7, 14)
       navitoclax  I-BET 762  Apicidin  belinostat  panobinostat  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Serdemetan  Sapanisertib  Temsirolimus  Deforolimus
BCL         0.002      1.000       1.0       1.000         1.000       1.000   1.000      1.000    1.000          1.0         1.0           1.0           1.0        1.000
BRD         1.000      0.034       1.0       1.000         1.000       1.000   1.000      1.000    1.000          1.0         1.0           1.0           1.0        1.000
HDAC        1.000      1.000       1.0       0.213         0.333       1.000   1.000      1.000    1.000          1.0         1.0           1.0           1.0        1.000
IGF1R       1.000      1.000       1.0       1.000         1.000       0.079   1.000      1.000    1.000          1.0         1.0           1.0           1.0        1.000
JAK         1.000      1.000       1.0       1.000         1.000       1.000   0.038      0.041    0.099      

In [4]:
pert_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        pert_matrix_top4_bottom4[i, j] = 1

fit_mask_top4_bottom4 = pert_matrix_top4_bottom4 * valid_matrix_top4_bottom4
print(f"pert_matrix shape: {pert_matrix_top4_bottom4.shape}")
print(pd.DataFrame(pert_matrix_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())
print(f"\nfit_mask (excludes missing-IC50 pairs):")
print(pd.DataFrame(fit_mask_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())


pert_matrix shape: (7, 14)
       navitoclax  I-BET 762  Apicidin  belinostat  panobinostat  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Serdemetan  Sapanisertib  Temsirolimus  Deforolimus
BCL             1          0         0           0             0           0       0          0        0            0           0             0             0            0
BRD             0          1         0           0             0           0       0          0        0            0           0             0             0            0
HDAC            0          0         1           1             1           0       0          0        0            0           0             0             0            0
IGF1R           0          0         0           0             0           1       0          0        0            0           0             0             0            0
JAK             0          0         0           0             0           0       1          1        1            1 

In [ ]:
residuals, a_coeffs = cstarpy.integration.pathway_activity.prediction.predict_coeffs(
    x, y_true_top4_bottom4, fit_mask_top4_bottom4,
    200_000, 10, 10, 10, 100
)

a_coeffs_df_top4_bottom4 = pd.DataFrame(a_coeffs, index=modules, columns=genes)
a_coeffs_df_top4_bottom4.to_csv(os.path.join(out_dir, "a_coeffs_df_top4_bottom4_btla.csv"))
print(f"a_coeffs shape: {a_coeffs.shape}")
trh = 0.001

print("\nGenes representing each module:")
print((abs(a_coeffs_df_top4_bottom4) > trh).sum(axis="columns").to_string())


100%|██████████| 200000/200000 [03:57<00:00, 842.96it/s] 


a_coeffs shape: (7, 15045)

Genes representing each module:
BCL       2
BRD       2
HDAC      4
IGF1R     2
JAK      26
MDM2      0
MTOR      2


In [13]:
trh = 0.0001

print("\nGenes representing each module:")
print((abs(a_coeffs_df_top4_bottom4) > trh).sum(axis="columns").to_string())



Genes representing each module:
BCL       2
BRD       2
HDAC      4
IGF1R     2
JAK      26
MDM2      0
MTOR      2


In [6]:
a_coeffs_top4_bottom4 = a_coeffs_df_top4_bottom4.values

pathway_activity_top4_bottom4 = a_coeffs_top4_bottom4 @ x
pd.DataFrame(pathway_activity_top4_bottom4, index=modules, columns=exp_list)\
    .to_csv(os.path.join(out_dir, "pathway_activity_top4_bottom4_btla.csv"))

R_global_top4_bottom4 = cstarpy.integration.pathway_activity.calc_global_response_from_pathway_activity(
    cstarpy.integration.pathway_activity.calc_pathway_activity(x, a_coeffs_top4_bottom4),
    modules, exp_list
)
R_global_df_top4_bottom4 = pd.DataFrame(R_global_top4_bottom4, index=modules, columns=exp_list)
R_global_df_top4_bottom4.to_csv(os.path.join(out_dir, "R_global_core_top4_bottom4_btla.csv"))

pd.DataFrame(y_true_top4_bottom4,      index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "y_true_top4_bottom4_btla.csv"))
pd.DataFrame(pert_matrix_top4_bottom4, index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "pert_matrix_top4_bottom4_btla.csv"))
x_df.to_csv(os.path.join(out_dir, "Data_norm_btla.csv"))


In [7]:
R_global_df_top4_bottom4

,navitoclax,I-BET 762,Apicidin,belinostat,panobinostat,BMS-536924,AT9283,TG-101348,CYT-387,ruxolitinib,Serdemetan,Sapanisertib,Temsirolimus,Deforolimus
BCL,-1.597066,0.000052,0.000102,0.000349,0.001484,0.000986,0.000041,0.000204,0.000597,-0.000918,-0.002297,0.000367,0.000867,0.000020
BRD,-0.000266,-1.549881,-0.000738,-0.716982,-0.002143,-0.002041,-0.000076,-0.682825,-0.115414,-0.474897,0.150208,-0.521081,0.086140,0.184871
HDAC,-0.000501,-0.003557,-0.000332,-1.014287,-0.967473,-0.004668,0.000055,-0.002360,-0.002967,-0.005960,-0.007929,-0.083249,-0.080402,-0.000548
IGF1R,-0.000091,-0.003506,-0.000747,-0.002914,-0.004536,-1.436457,-0.000012,-0.002574,-0.003046,-0.010474,-0.003072,-0.006211,-0.003560,-0.002098
JAK,0.000200,-1.971294,-0.675490,-1.305855,-1.251029,-1.987072,-1.146324,-1.492864,-1.297987,-1.937302,-0.731294,-1.825093,-1.253833,-0.004630
MDM2,-0.000080,-0.003555,0.000179,0.000015,0.001095,-0.001524,-0.000131,-0.000630,0.000234,-0.002834,-0.001708,-0.002714,-0.000257,-0.000408
MTOR,0.000047,-0.002098,-0.000050,-0.000969,-0.002388,-0.002688,-0.000074,-0.001610,-0.002400,-0.010723,-0.010534,-1.957584,-1.913221,-1.629238


In [8]:
pert_matrix_top4_bottom4

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.]])